# 02 — Chính sách chuẩn bị dữ liệu RFM + Engagement

**P2-T2 PASS theo xác nhận người dùng; P2-T3 READY FOR STUDENT REVIEW.**

**Câu hỏi:** giữ cột và event nào để năm biến có ý nghĩa, đồng thời không làm mất tương tác hợp lệ?

**Lý do:** category_code thiếu 98,33%, brand thiếu 41,92% nhưng không phục vụ năm biến. Không cần loại event vì hai trường này. Product/category vẫn phân biệt event khi loại trùng.

Notebook đọc kết quả kiểm chứng toàn bộ bốn CSV; bảng lớn chỉ tồn tại tạm trong DuckDB. Không xuất observation chính thức hoặc bảng feature, không K-Means. Output các ô dưới được chạy bằng Python local, chưa xác nhận lần mới trên Colab. Lý thuyết chi tiết: [sổ tay phần 19](../docs/LEARNING_GUIDE.md).

## 1. Chuẩn bị bản thực hành

Mở **bản repo đầy đủ** từ root hoặc notebooks. Nếu dùng Colab, clone đúng nhánh/commit được nhóm chọn rồi đổi cwd vào `/content/customer-segmentation-mvp`; notebook tải riêng không kèm results/scripts. Thành viên 2 sửa chính, Thành viên 1 kiểm tra trên bản Drive riêng; restart và chạy từ đầu. Chỉ cần thư viện chuẩn để đọc kết quả. Bật RUN_POLICY khi đã có bốn CSV raw, DuckDB và REES46_DATA_ROOT; output mới giúp giữ lần trước.


In [1]:
import csv
import hashlib
import json
import os
from decimal import Decimal
from pathlib import Path
import subprocess
import sys

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/check_cleaning_policy.py").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Cần clone repo và chuyển cwd vào root hoặc notebooks.")
RUN_POLICY = False
RESULTS = REPO / "results/p2_t3"
if RUN_POLICY:
    if not os.environ.get("REES46_DATA_ROOT"):
        raise ValueError("Đặt REES46_DATA_ROOT tới thư mục chứa raw trước.")
    RESULTS = REPO / "results/p2_t3_recheck"
    subprocess.run([sys.executable, "-B", "scripts/check_cleaning_policy.py",
                    "--output", str(RESULTS)], cwd=REPO, check=True)
def read_rows(path):
    with path.open(encoding="utf-8", newline="") as stream:
        return list(csv.DictReader(stream))
stages = read_rows(RESULTS / "stage_summary.csv")
counts = {r["metric"]: int(r["count"]) for r in read_rows(RESULTS / "policy_counts.csv")}
manifest = json.loads((RESULTS / "run_manifest.json").read_text(encoding="utf-8"))
print("Đã đọc hai CSV và manifest của", manifest["task"])


Đã đọc hai CSV và manifest của P2-T3


## 2. Chính sách được đề nghị

| Bước | Cách làm và lý do |
|---|---|
| Duplicate | Một dòng/chín giá trị văn bản gốc; chưa cast/trim. Giữ product/category trong khóa. Thiếu event_id nên đây là giả định cần sinh viên chấp nhận. |
| Timestamp/user | Cần timezone rõ, parse hữu hạn, trong observation UTC; user là BIGINT dương. Hàng sai không dùng cho năm biến; đếm lý do loại riêng, có thứ tự. |
| Session | Thiếu/sai dạng UUID thành NULL, không tạo phiên giả. Ghép user-session. Giữ view có user/time hợp lệ dù thiếu phiên. |
| Price/buyer | Giữ event giá âm/0/thiếu; R/F/M chỉ dùng purchase giá dương, phiên hợp lệ. Giá cao không tự là lỗi. Buyer có ít nhất một purchase hợp lệ. |
| Rounding | Guard toàn bộ price trước DECIMAL(18,2); gặp độ chính xác khác phải dừng. R giữ phần lẻ, số đếm giữ nguyên; chỉ làm tròn khi hiển thị. |
| Output | Chọn event_time, event_type, user_id, user_session, price. Bỏ bốn cột còn lại cuối cùng, không DISTINCT lần nữa. |

Ví dụ minh họa: cùng phiên mua hai sản phẩm khác nhau, mỗi sản phẩm giá 10, thì F=1 và M=20. Bỏ product trước dedup có thể làm M còn 10. Trong bảng sau, Sessions/Views là tổng trên **toàn users**, chưa phải đặc trưng từng buyer. RFM dùng observation, Engagement dùng `[02/01/2020,01/02/2020)` theo UTC.


In [2]:
before, after = stages
print(f"{'Chỉ tiêu':38} {'Trước dedup':>16} {'Sau chính sách':>16} {'Chênh lệch':>16}")
for metric in before:
    if metric == "stage":
        continue
    first, last = Decimal(before[metric]), Decimal(after[metric])
    print(f"{metric:38} {str(first):>16} {str(last):>16} {str(last-first):>16}")


Chỉ tiêu                                    Trước dedup   Sau chính sách       Chênh lệch
event_rows                                     16536158         15667350          -868808
view_events                                     7704235          7703765             -470
purchase_events                                 1045014          1044221             -793
eligible_purchase_events                        1044922          1044133             -789
invalid_price_purchase_events                        92               88               -4
events_without_session                             3543             3179             -364
eligible_buyers                                   92652            92652                0
all_purchase_sessions                            126503           126503                0
purchase_sessions                                126503           126503                0
all_purchase_price_sum                       5141715.66       5138624.02         -3091.64
purchase_p

**Đọc kết quả:** duplicate làm giảm số dòng, tổng price mua dương và số view; không đổi số buyers/phiên mua/phiên 30 ngày trong dữ liệu này. Giữ 88 purchase giá không dương trong event sau dedup, nhưng không đưa chúng vào R/F/M. Không có user mất toàn bộ purchase hợp lệ hoặc đổi thời điểm mua cuối trong lần kiểm tra này; đó là kết quả dữ liệu hiện tại, không phải quy tắc luôn đúng.

## 3. Hàng bị loại và ảnh hưởng của projection

Các lý do loại sau dedup không giao nhau: timestamp sai → ngoài observation → user sai → loại event lạ. Các cờ session/price chỉ giới hạn phép tính tương ứng, không cộng thêm vào số hàng bị xóa.


In [3]:
for metric, value in counts.items():
    print(f"{metric}: {value:,}")
print()
print("Phiên có purchase nhưng không có purchase hợp lệ:",
      int(after["all_purchase_sessions"]) - int(after["purchase_sessions"]))
print("Tác động lên tổng price nếu chỉ cộng purchase hợp lệ:",
      Decimal(after["purchase_price_sum"]) - Decimal(after["all_purchase_price_sum"]))


raw_rows: 16,536,158
duplicate_extras_removed: 868,808
invalid_timestamp: 0
outside_observation: 0
invalid_user: 0
unknown_event_type: 0
output_rows_dry_run: 15,667,350
distinct_raw_events_sharing_projection: 363,403
users_no_eligible_purchase: 0
buyers_last_purchase_changed: 0

Phiên có purchase nhưng không có purchase hợp lệ: 0
Tác động lên tổng price nếu chỉ cộng purchase hợp lệ: 2642.88


**Nhận xét:** 363.403 dòng sẽ bị mất nếu lại DISTINCT trên năm cột, dù chúng khác nhau trên chín trường nguồn. Giữ chúng theo chính sách; không suy rằng đã chứng minh mọi event nguồn đều là hành vi thật. Tám view trong cửa sổ 30 ngày thiếu session vẫn được tính Views, không được tính thành một phiên NULL.

## 4. Schema, kiểm tra và giới hạn

Năm cột đủ cho bước tính sau; DuckDB đọc Parquet có thể chỉ lấy cột cần dùng. Task này chỉ kiểm thử Parquet bằng fixture nhỏ riêng, chưa xuất observation chính thức. DECIMAL giúp tổng price không bị sai số cộng số thực; đây không phải khẳng định currency hay net GMV.


In [4]:
removed = counts["duplicate_extras_removed"] + sum(counts[k] for k in
    ["invalid_timestamp", "outside_observation", "invalid_user", "unknown_event_type"])
assert counts["raw_rows"] == removed + counts["output_rows_dry_run"]
assert int(after["event_rows"]) == counts["output_rows_dry_run"]
assert int(after["eligible_purchase_events"]) <= int(after["purchase_events"])
assert int(after["purchase_sessions"]) <= int(after["all_purchase_sessions"])
assert manifest["input_metadata_unchanged"] and manifest["price_precision_guard_passed"]
script = REPO / "scripts/check_cleaning_policy.py"
source_bytes = script.read_bytes()
assert manifest["code_sha256"] in {
    hashlib.sha256(source_bytes).hexdigest(),
    hashlib.sha256(source_bytes.replace(b"\r\n", b"\n")).hexdigest()}
assert [c["column"] for c in manifest["output_schema"]] == [
    "event_time", "event_type", "user_id", "user_session", "price"]
for col in manifest["output_schema"]:
    print(col["column"], "→", col["dtype"])
print("Đối chiếu số dòng, schema, hash code và manifest đạt.")


event_time → TIMESTAMP WITH TIME ZONE
event_type → VARCHAR
user_id → BIGINT
user_session → VARCHAR
price → DECIMAL(18,2)
Đối chiếu số dòng, schema, hash code và manifest đạt.


**Giới hạn:** hash local/metadata ổn định chưa xác thực nguồn tải. Không có event_id để phân biệt mọi lặp kỹ thuật với hành vi. Giá cao, refunds/quantity/currency và môi trường cài mới vẫn cần thận trọng. February đã được xem ở dự án cũ, không phải untouched holdout; không đọc events tháng này. Chưa chạy Colab mới, chưa có model hay bằng chứng Engagement cải thiện kết quả.

**Tự kiểm tra:** Vì sao chỉ bỏ product/category cuối cùng? Purchase âm có còn xác nhận phiên tương tác không? Khi price có ba chữ số thập phân, vì sao phải dừng thay vì tự round?

Dừng **READY FOR STUDENT REVIEW**, không tự chuyển P2-T4. Không thêm hình vì biểu đồ duplicate trong notebook 01 đã thể hiện thay đổi lớn.
